# Day 024 — Exercise 2: extract_schema_fields

**What you'll build:** `extract_schema_fields(text, fields, model)` — gives the LLM a list of field names and plain text, and gets back a dict with one value per field.

**Why it matters:** This is schema-guided extraction without Pydantic. The field list acts as a mini-schema. `format='json'` (Day 4) guarantees parseable output. Null values for unfound fields and a fallback dict on parse failure make it resilient.

In [ ]:
import json
import ollama

## Your Implementation

In [ ]:
def extract_schema_fields(
    text: str, fields: list[str], model: str = "llama3.2"
) -> dict:
    """
    Extract named fields from plain text using the LLM.

    Args:
        text:   Plain text to extract from (already stripped of HTML).
        fields: List of field names to extract (e.g. ['title', 'price']).
        model:  Ollama model name.

    Returns:
        Dict with exactly the requested field names as keys.
        Missing fields are null. Never raises.
    """
    # TODO: fields_json = json.dumps(fields)
    # TODO: call ollama.chat with format='json'
    #       system: 'extract these fields: {fields_json}; use null for missing'
    #       user:   f'Extract from this text:\n\n{text[:3000]}'
    # TODO: raw = response['message']['content']
    # TODO: try: return json.loads(raw)
    #       except: return {f: None for f in fields}
    pass

## Check Your Work

In [ ]:
SAMPLE_TEXT = 'Python is a high-level programming language created by Guido van Rossum and first released in 1991. It emphasizes code readability and simplicity. Python is widely used in data science, web development, and automation.'
FIELDS = ['language_name', 'creator', 'year']


def _run_checks():
    total = 5
    passed = 0

    # Check 1: defined
    try:
        assert 'extract_schema_fields' in globals()
        passed += 1; print('\u2705 Check 1: extract_schema_fields defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}')
        return

    result = None

    # Check 2: returns a dict (1 LLM call)
    try:
        result = extract_schema_fields(SAMPLE_TEXT, FIELDS)
        assert isinstance(result, dict), f'expected dict, got {type(result)}'
        passed += 1; print('\u2705 Check 2: returns a dict')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: dict has all requested field keys
    try:
        assert result is not None, 'result is None (Check 2 failed)'
        for f in FIELDS:
            assert f in result, f"missing field '{f}': {list(result)}'"
        passed += 1; print(f'\u2705 Check 3: dict has all {len(FIELDS)} requested fields')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: at least one field has a non-null string value
    try:
        assert result is not None, 'result is None'
        non_null = [v for v in result.values() if v is not None]
        assert len(non_null) >= 1, \
            f'all fields are null — LLM likely not extracting: {result}'
        passed += 1; print(f'\u2705 Check 4: {len(non_null)}/{len(FIELDS)} fields extracted')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: works on empty text without crashing (1 LLM call)
    try:
        empty_result = extract_schema_fields('', ['name', 'date'])
        assert isinstance(empty_result, dict), \
            f'expected dict for empty text, got {type(empty_result)}'
        assert 'name' in empty_result and 'date' in empty_result, \
            f"fallback keys missing: {list(empty_result)}"
        passed += 1; print('\u2705 Check 5: works on empty text — returns dict with requested keys')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def extract_schema_fields(
    text: str, fields: list[str], model: str = "llama3.2"
) -> dict:
    fields_json = json.dumps(fields)
    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a structured data extractor. "
                    f"Extract the following fields from the text: {fields_json}. "
                    "Return JSON with exactly these keys. "
                    "Use null for any field you cannot find. "
                    "Return only valid JSON, no explanation."
                ),
            },
            {
                "role": "user",
                "content": f"Extract from this text:\n\n{text[:3000]}",
            },
        ],
        format="json",
    )
    raw = response["message"]["content"]
    try:
        return json.loads(raw)
    except Exception:
        return {f: None for f in fields}
```

</details>